In [1]:
import h5py
import pandas as pd 
import xarray as xr
import os
import cfgrib
from pathlib import Path
from dotenv import load_dotenv

home_dir = os.getenv("HOME_FOLDER")

Let the two filesbe separate as both of them have two different time periodicity. Will merge after averaging out perdaily.

Grib file was hard to look at, so converted them to netcdf files. Merging them to one zarr file 

In [ ]:
datasets = cfgrib.open_datasets(
    f"{home_dir}/data/raw/era5/era5_2025_2.grib"        # specify the file path to GRIB file
)

for ds in datasets:

    vars_name = "_".join(ds.data_vars)

    ds.to_netcdf(
        f"2025_2_{vars_name}.nc"                           #Rename the output file based on the file and time period 
    )

The problem was that the frequency of measurement was different for 2 sets of variables(ssrd e and tp vs the rest)
So solution was to merge ehem separatelty store them separately and merge them later after averaging daily 

In [ ]:
ds1 = xr.open_dataset("../data/intermediate/era5/2025_1_sp_blh_tcc_u10_v10_t2m_d2m_skt.nc", chunks="auto")

ds2 = xr.open_dataset("../data/raw/era5/2025_2_sp_blh_tcc_u10_v10_t2m_d2m_skt.nc", chunks="auto")


master = xr.concat([ds1, ds2], dim="time", compat="override", coords="minimal")  # Concatenate the merged datasets along the time dimension

<xarray.Dataset> Size: 2GB
Dimensions:     (time: 4344, latitude: 127, longitude: 119)
Coordinates:
  * time        (time) datetime64[ns] 35kB 2025-01-01 ... 2025-06-30T23:00:00
    valid_time  (time) datetime64[ns] 35kB dask.array<chunksize=(4344,), meta=np.ndarray>
  * latitude    (latitude) float64 1kB 37.5 37.25 37.0 36.75 ... 6.5 6.25 6.0
  * longitude   (longitude) float64 952B 68.0 68.25 68.5 ... 97.0 97.25 97.5
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Data variables:
    sp          (time, latitude, longitude) float32 263MB dask.array<chunksize=(3495, 101, 95), meta=np.ndarray>
    blh         (time, latitude, longitude) float32 263MB dask.array<chunksize=(3495, 101, 95), meta=np.ndarray>
    tcc         (time, latitude, longitude) float32 263MB dask.array<chunksize=(3495, 101, 95), meta=np.ndarray>
    u10         (time, latitude, longitude) float32 263MB dask.array<chunksize=(3495, 101, 95), meta=np.ndarray>
    v10   

/tmp/ipykernel_3537/3916706583.py:5: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  merged1 = xr.merge([ds1a, ds1b], compat="override")  # Merge the datasets, overriding any conflicting variables


FileNotFoundError: [Errno 2] No such file or directory: '/home/victus7/Desktop/Personal-Project/AirPollution/data/raw/era5/2025_2_sp_blh_tcc_u10_v10_t2m_d2m_skt.nc'

In [19]:
master = master.chunk({
    "time" : 24,
    "latitude" : 50,
    "longitude" : 50
})

master.to_zarr("../data/processed/era5/era5_2025.zarr", mode="w")  # Save the final dataset as a Zarr file

/home/victus7/Desktop/Personal-Project/AirPollution/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


And the surface variable extraction is below 

In [19]:
ds1 = xr.open_dataset("../data/intermediate/era5/2025_1_ssrd_e_tp.nc", chunks="auto")
ds2 = xr.open_dataset("../data/intermediate/era5/2025_2_ssrd_e_tp.nc", chunks="auto")


master = xr.concat([ds1, ds2], dim="time", compat="override", coords="minimal")  # Concatenate the merged datasets along the time dimension

master["e"] = master["e"].chunk("auto")
master["ssrd"] = master["ssrd"].chunk("auto")
master["tp"] = master["tp"].chunk("auto")
master["valid_time"] = master["valid_time"].chunk("auto")

master = master.chunk({
    "time" : 24,
    "latitude" : -1,
    "longitude" : -1
})


master.to_zarr("../data/processed/era5_surf/era5_surf_2025.zarr", mode="w")  # Save the final dataset as a Zarr file

/home/victus7/Desktop/Personal-Project/AirPollution/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
